## Part 1: Setup and Configuration

In [6]:
import pandas as pd
import numpy as np
import joblib
import os
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

## Part 2: Data Loading and Configuration

In [7]:
DATA_FILE = r"C:\Users\manju\OneDrive\Desktop\Documents\Data Analytics\DA Semester 4\Capstone 2\ML Modelling\ml_dataset.csv"
OUTPUT_DIR = './models'

# Create output directory if it does not exist
Path(OUTPUT_DIR).mkdir(exist_ok=True)

# Load dataset
df = pd.read_csv(DATA_FILE)

print(f'Dataset loaded: {df.shape[0]:,} rows, {df.shape[1]:,} columns')
print(f'Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB')

Dataset loaded: 69,555 rows, 51 columns
Memory usage: 46.20 MB


In [8]:
# Rename columns for consistency
df.rename(columns={'Other': 'other_healthcare_occupations_wages'}, inplace=True)

## Part 3: Feature Definition

In [9]:
# Define all features used in the model
# These features were selected based on domain knowledge and do not cause data leakage

NUMERICAL_FEATURES = [
    'year',
    'number_of_beds',
    'total_bed_days_available',
    'occupancy_rate',
    'total_discharges__v___xviii___xix___unknown_',
    'total_days__v___xviii___xix___unknown_',
    'fte___employees_on_payroll',
    'staff_to_bed_ratio',
    'revenue_per_bed',
    'discharges_per_bed',
    'avg_length_of_stay',
    'total_other_expenses',
    'rucc_code',
    'wage_physician',
    'wage_practitioner',
    'wage_rn',
    'wage_support',
    'other_healthcare_occupations_wages',
    'population',
    'median_age',
    'median_income',
    'poverty_rate',
    'higher_education_rate',
]

CATEGORICAL_FEATURES = [
    'provider_type',
    'ccn_facility_type',
    'type_of_control',
    'rural_urban',
]

ALL_FEATURES = NUMERICAL_FEATURES + CATEGORICAL_FEATURES
TARGET_PM = 'profit_margin_calc'
TARGET_CCR = 'charity_care_ratio'

# Verify that all features exist in the dataset
missing_features = [f for f in ALL_FEATURES if f not in df.columns]
if missing_features:
    print(f'ERROR: Missing columns: {missing_features}')
    raise ValueError(f'Dataset missing required features: {missing_features}')

print(f'Feature verification passed.')
print(f'  Total features: {len(ALL_FEATURES)}')
print(f'  Numerical: {len(NUMERICAL_FEATURES)}')
print(f'  Categorical: {len(CATEGORICAL_FEATURES)}')

Feature verification passed.
  Total features: 27
  Numerical: 23
  Categorical: 4


## Part 4: Data Preparation

In [10]:
# Check for missing values in critical columns
print('Missing values analysis:')
missing = df[ALL_FEATURES + [TARGET_PM, TARGET_CCR]].isnull().sum()
if missing.sum() > 0:
    print(missing[missing > 0])
else:
    print('  No missing values found.')

# Remove rows with missing target values
df_clean = df.dropna(subset=[TARGET_PM, TARGET_CCR]).copy()

print(f'\nDataset after cleaning: {df_clean.shape[0]:,} rows')
print(f'Removed: {df.shape[0] - df_clean.shape[0]} rows with missing targets')

Missing values analysis:
  No missing values found.

Dataset after cleaning: 69,555 rows
Removed: 0 rows with missing targets


## Part 5: Train-Test Split

In [11]:
# Extract features and targets
X = df_clean[ALL_FEATURES].copy()
y_pm = df_clean[TARGET_PM].copy()
y_ccr = df_clean[TARGET_CCR].copy()

# Split into training (80%) and testing (20%)
# Use random_state=42 for reproducibility
X_train, X_test, y_pm_train, y_pm_test = train_test_split(
    X, y_pm,
    test_size=0.2,
    random_state=42
)

# Align charity care ratio labels with the same train/test split
y_ccr_train = y_ccr.loc[y_pm_train.index]
y_ccr_test = y_ccr.loc[y_pm_test.index]

print(f'Training set: {X_train.shape[0]:,} rows')
print(f'Testing set: {X_test.shape[0]:,} rows')
print(f'\nTarget distributions:')
print(f'Profit Margin - Train: mean={y_pm_train.mean():.4f}, std={y_pm_train.std():.4f}')
print(f'Profit Margin - Test: mean={y_pm_test.mean():.4f}, std={y_pm_test.std():.4f}')
print(f'Charity Care Ratio - Train: mean={y_ccr_train.mean():.4f}, std={y_ccr_train.std():.4f}')
print(f'Charity Care Ratio - Test: mean={y_ccr_test.mean():.4f}, std={y_ccr_test.std():.4f}')

Training set: 55,644 rows
Testing set: 13,911 rows

Target distributions:
Profit Margin - Train: mean=0.0394, std=0.1828
Profit Margin - Test: mean=0.0372, std=0.1839
Charity Care Ratio - Train: mean=0.0256, std=0.0290
Charity Care Ratio - Test: mean=0.0259, std=0.0291


## Part 6: Pipeline Construction

In [12]:
def build_preprocessing_pipeline():
    """
    Build the preprocessing pipeline for feature transformation.
    
    Returns:
        ColumnTransformer: Preprocessor that handles categorical and numerical features
    """
    preprocessor = ColumnTransformer(
        transformers=[
            ('categorical', 
             OneHotEncoder(handle_unknown='ignore', sparse_output=False),
             CATEGORICAL_FEATURES),
            ('numerical',
             SimpleImputer(strategy='median'),
             NUMERICAL_FEATURES)
        ]
    )
    return preprocessor


def build_model_pipeline(model):
    """
    Build the complete machine learning pipeline.
    
    Parameters:
        model: The estimator to use in the pipeline
    
    Returns:
        Pipeline: Complete preprocessing + model pipeline
    """
    pipeline = Pipeline([
        ('preprocessor', build_preprocessing_pipeline()),
        ('model', model)
    ])
    return pipeline


print('Pipeline functions defined.')

Pipeline functions defined.


## Part 7: Model 1 - Profit Margin Prediction (Random Forest)

In [ ]:
print('Training Model 1: Random Forest for Profit Margin Prediction')
print('/n')
# Create Random Forest model with optimized hyperparameters
rf_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=2,
    min_samples_split=2,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1,
    verbose=0
)

# Build complete pipeline
pipe_pm = build_model_pipeline(rf_model)

# Train on training data
pipe_pm.fit(X_train, y_pm_train)

# Make predictions
y_pm_pred_train = pipe_pm.predict(X_train)
y_pm_pred_test = pipe_pm.predict(X_test)

# Calculate metrics
pm_train_r2 = r2_score(y_pm_train, y_pm_pred_train)
pm_test_r2 = r2_score(y_pm_test, y_pm_pred_test)
pm_mae = mean_absolute_error(y_pm_test, y_pm_pred_test)
pm_rmse = np.sqrt(mean_squared_error(y_pm_test, y_pm_pred_test))

print(f'\nProfit Margin Model Results:')
print(f'  Training R2:  {pm_train_r2:.4f}')
print(f'  Testing R2:   {pm_test_r2:.4f}')
print(f'  Test MAE:     {pm_mae:.4f}')
print(f'  Test RMSE:    {pm_rmse:.4f}')

# Check for overfitting
overfitting_gap = pm_train_r2 - pm_test_r2
print(f'  Overfitting gap: {overfitting_gap:.4f}')

Training Model 1: Random Forest for Profit Margin Prediction
This may take 2-3 minutes...
------------------------------------------------------------

Profit Margin Model Results:
  Training R2:  0.8311
  Testing R2:   0.4060
  Test MAE:     0.0849
  Test RMSE:    0.1417
  Overfitting gap: 0.4251


## Part 8: Model 1 - Cross-Validation

In [14]:
print('5-fold cross-validation for Profit Margin model')

kf = KFold(n_splits=5, shuffle=True, random_state=42)

cv_scores_pm = cross_val_score(
    build_model_pipeline(
        RandomForestRegressor(
            n_estimators=300,
            max_depth=None,
            min_samples_leaf=2,
            min_samples_split=2,
            max_features='sqrt',
            random_state=42,
            n_jobs=-1
        )
    ),
    X_train, y_pm_train,
    cv=kf,
    scoring='r2',
    n_jobs=-1
)

print(f'Cross-validation R2 scores: {[round(s, 4) for s in cv_scores_pm]}')
print(f'Mean CV R2: {cv_scores_pm.mean():.4f}')
print(f'Std CV R2: {cv_scores_pm.std():.4f}')

Running 5-fold cross-validation for Profit Margin model...
Cross-validation R2 scores: [np.float64(0.4029), np.float64(0.3799), np.float64(0.3761), np.float64(0.4039), np.float64(0.3799)]
Mean CV R2: 0.3886
Std CV R2: 0.0122


## Part 9: Model 2 - Charity Care Ratio Prediction (ExtraTrees)

In [ ]:
print('Model 2: ExtraTrees for Charity Care Ratio Prediction')
print("/n")

# Create ExtraTrees model with optimized hyperparameters
et_model = ExtraTreesRegressor(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=1,
    min_samples_split=2,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1,
    verbose=0
)

# Build complete pipeline
pipe_ccr = build_model_pipeline(et_model)

# Train on training data
pipe_ccr.fit(X_train, y_ccr_train)

# Make predictions
y_ccr_pred_train = pipe_ccr.predict(X_train)
y_ccr_pred_test = pipe_ccr.predict(X_test)

# Calculate metrics
ccr_train_r2 = r2_score(y_ccr_train, y_ccr_pred_train)
ccr_test_r2 = r2_score(y_ccr_test, y_ccr_pred_test)
ccr_mae = mean_absolute_error(y_ccr_test, y_ccr_pred_test)
ccr_rmse = np.sqrt(mean_squared_error(y_ccr_test, y_ccr_pred_test))

print(f'\nCharity Care Ratio Model Results:')
print(f'  Training R2:  {ccr_train_r2:.4f}')
print(f'  Testing R2:   {ccr_test_r2:.4f}')
print(f'  Test MAE:     {ccr_mae:.4f}')
print(f'  Test RMSE:    {ccr_rmse:.4f}')

# Check for overfitting
overfitting_gap = ccr_train_r2 - ccr_test_r2
print(f'  Overfitting gap: {overfitting_gap:.4f}')

Model 2: ExtraTrees for Charity Care Ratio Prediction
This may take 2-3 minutes...
------------------------------------------------------------

Charity Care Ratio Model Results:
  Training R2:  1.0000
  Testing R2:   0.5947
  Test MAE:     0.0102
  Test RMSE:    0.0185
  Overfitting gap: 0.4053


## Part 10: Model 2 - Cross-Validation

In [16]:
print('5-fold cross-validation for Charity Care Ratio model')

cv_scores_ccr = cross_val_score(
    build_model_pipeline(
        ExtraTreesRegressor(
            n_estimators=300,
            max_depth=None,
            min_samples_leaf=1,
            min_samples_split=2,
            max_features='sqrt',
            random_state=42,
            n_jobs=-1
        )
    ),
    X_train, y_ccr_train,
    cv=kf,
    scoring='r2',
    n_jobs=-1
)

print(f'Cross-validation R2 scores: {[round(s, 4) for s in cv_scores_ccr]}')
print(f'Mean CV R2: {cv_scores_ccr.mean():.4f}')
print(f'Std CV R2: {cv_scores_ccr.std():.4f}')

5-fold cross-validation for Charity Care Ratio model
Cross-validation R2 scores: [np.float64(0.5648), np.float64(0.5546), np.float64(0.5556), np.float64(0.5648), np.float64(0.5465)]
Mean CV R2: 0.5573
Std CV R2: 0.0069


## Part 11: Results Summary

In [ ]:
# Create comprehensive results summary
results_df = pd.DataFrame({
    'Target': ['Profit Margin', 'Charity Care Ratio'],
    'Model': ['Random Forest (300 trees)', 'ExtraTrees (300 trees)'],
    'Train R2': [round(pm_train_r2, 4), round(ccr_train_r2, 4)],
    'Test R2': [round(pm_test_r2, 4), round(ccr_test_r2, 4)],
    'CV R2 Mean': [round(cv_scores_pm.mean(), 4), round(cv_scores_ccr.mean(), 4)],
    'CV R2 Std': [round(cv_scores_pm.std(), 4), round(cv_scores_ccr.std(), 4)],
    'Test MAE': [round(pm_mae, 4), round(ccr_mae, 4)],
    'Test RMSE': [round(pm_rmse, 4), round(ccr_rmse, 4)],
})

print('\n')
print('FINAL MODEL RESULTS SUMMARY')
print('/n')
print(results_df.to_string(index=False))
print('/n')


FINAL MODEL RESULTS SUMMARY
            Target                     Model  Train R2  Test R2  CV R2 Mean  CV R2 Std  Test MAE  Test RMSE
     Profit Margin Random Forest (300 trees)    0.8311   0.4060      0.3886     0.0122    0.0849     0.1417
Charity Care Ratio    ExtraTrees (300 trees)    1.0000   0.5947      0.5573     0.0069    0.0102     0.0185


## Part 12: Save Models

In [18]:
# Define output file paths
pm_model_path = os.path.join(OUTPUT_DIR, 'profit_margin_model.pkl')
ccr_model_path = os.path.join(OUTPUT_DIR, 'charity_care_model.pkl')
metadata_path = os.path.join(OUTPUT_DIR, 'feature_metadata.pkl')

# Save trained models
joblib.dump(pipe_pm, pm_model_path)
joblib.dump(pipe_ccr, ccr_model_path)

# Save feature metadata for use in Flask app
metadata = {
    'numerical_features': NUMERICAL_FEATURES,
    'categorical_features': CATEGORICAL_FEATURES,
    'all_features': ALL_FEATURES,
    'target_pm': TARGET_PM,
    'target_ccr': TARGET_CCR,
}
joblib.dump(metadata, metadata_path)

print('Models saved successfully:')
print(f'  Profit Margin model: {pm_model_path}')
print(f'  Charity Care model:  {ccr_model_path}')
print(f'  Feature metadata:    {metadata_path}')
print(f'\nOutput directory: {os.path.abspath(OUTPUT_DIR)}')

Models saved successfully:
  Profit Margin model: ./models\profit_margin_model.pkl
  Charity Care model:  ./models\charity_care_model.pkl
  Feature metadata:    ./models\feature_metadata.pkl

Output directory: c:\Users\manju\Downloads\models


## Part 13: Model Verification

In [19]:
# Load and verify saved models
print('Verifying saved models by loading and making predictions...')
print('-' * 60)

loaded_pm_model = joblib.load(pm_model_path)
loaded_ccr_model = joblib.load(ccr_model_path)
loaded_metadata = joblib.load(metadata_path)

# Get sample test data
sample_data = X_test.head(5).copy()

# Make predictions with loaded models
sample_pm_pred = loaded_pm_model.predict(sample_data)
sample_ccr_pred = loaded_ccr_model.predict(sample_data)

# Create verification dataframe
verification_df = pd.DataFrame({
    'Index': sample_data.index,
    'Actual PM': y_pm_test.head(5).values.round(4),
    'Predicted PM': sample_pm_pred.round(4),
    'Actual CCR': y_ccr_test.head(5).values.round(4),
    'Predicted CCR': sample_ccr_pred.round(4),
})

print('Sample predictions on first 5 test records:')
print(verification_df.to_string(index=False))

print('\nVerification complete - all models loaded and working correctly.')
print('\nFeature metadata loaded:')
print(f'  Numerical features: {len(loaded_metadata["numerical_features"])}')
print(f'  Categorical features: {len(loaded_metadata["categorical_features"])}')
print(f'  Total features: {len(loaded_metadata["all_features"])}')

Verifying saved models by loading and making predictions...
------------------------------------------------------------
Sample predictions on first 5 test records:
 Index  Actual PM  Predicted PM  Actual CCR  Predicted CCR
 42416    -0.1568        0.0641      0.0147         0.0189
 34965    -0.4926        0.0564      0.0090         0.0145
 37281    -0.0332        0.0352      0.0028         0.0129
 10276     0.1262        0.0157      0.0166         0.0365
 62536     0.0817        0.0786      0.0209         0.0180

Verification complete - all models loaded and working correctly.

Feature metadata loaded:
  Numerical features: 23
  Categorical features: 4
  Total features: 27
